# KCS 2023年7/8月公开数值时间见证

冻结完整Stage1 0bp Tiered父方案，只核对官方原件数值与两篇当期新闻的发布日期。新闻当前HTML不是历史不可变快照。零 SOXX 收益、拟合、账户。

In [1]:
import datetime,hashlib,json,os,pathlib,re,shlex,subprocess,tarfile,tempfile,urllib.request
from bs4 import BeautifulSoup
import qlib
from qlib.data import D
from qlib.workflow import R
import sys
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
card=root/'research/kcs_early20_2023_jul_aug_public_value_witness_20261023';snap=root/'data/snapshots/kcs_early20_2023_jul_aug_public_value_witness_20261023';snap.mkdir(parents=True,exist_ok=True)
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
source=root/'research/kcs_early20_2023_2026_semiconductor_yoy_source_v2_20261023';assert sha(source/'summary.json')==plan['source_numeric_summary_sha256']
original=root/'data/snapshots/kcs_early20_2023_2026_held_original_capture_20261023';assert json.loads((root/'research/kcs_early20_2023_2026_held_original_capture_20261023/source_restore.json').read_text())['status']=='NAS_four_held_year_source_groups_all_members_cold_restored_exact'
manifest=json.loads((original/'2023/year_manifest.json').read_text());month_map={m['month']:m for m in manifest['months']}
for case in plan['fixed_cases']:
 m=month_map[case['month']];assert m['page']['newsId']==case['official_newsId'];a=m['page']['attachments'][0];assert a['body_sha256']==case['official_HWP_sha256'] and sha(root/a['local_path'])==a['body_sha256'] and (root/a['local_path']).stat().st_size==a['body_bytes']
state={'status':'capturing','plan_sha256':sha(card/'plan.json'),'GETs_started':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[]
for case in plan['fixed_cases']:
 month=case['month'];url=case['witness_url'];start=datetime.datetime.now(datetime.timezone.utc).isoformat();record={'month':month,'url':url,'started_utc':start,'http_status':None,'error':None,'final_url':None,'body_bytes':0,'body_sha256':None,'local_path':None}
 state['GETs_started']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 try:
  req=urllib.request.Request(url,headers={'User-Agent':'soxl-qlib-source-research'})
  with urllib.request.urlopen(req,timeout=20) as response:
   record['http_status']=response.status;record['final_url']=response.url;record['headers']={k:v for k,v in response.headers.items() if k.lower() in ('content-type','content-length','last-modified','date','etag')};body=response.read(500001)
  if len(body)>500000:raise ValueError('body exceeds frozen 500000 byte cap')
  p=snap/(month+'.response');p.write_bytes(body);record.update(body_bytes=len(body),body_sha256=sha(p),local_path=str(p.relative_to(root)))
 except Exception as e:record['error']=type(e).__name__+': '+str(e)[:250]
 record['ended_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat();receipts.append(record)
(snap/'response_receipts.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
calendar=D.calendar(start_time='2023-07-21',end_time='2023-08-31',freq='day');calendar_dates=[str(x)[:10] for x in calendar];cases=[]
for case,rec in zip(plan['fixed_cases'],receipts):
 month=case['month'];first=next((x for x in calendar_dates if x>case['witness_publication_date_KST']),None);w={'month':month,'official_reported_YoY_pct':case['expected_reported_export_YoY_pct'],'official_HWP_sha256':case['official_HWP_sha256'],'witness_url':case['witness_url'],'witness_body_sha256':rec['body_sha256'],'witness_HTTP_status':rec['http_status'],'first_US_Qlib_session_strictly_after_article_date':first,'publisher_initial_display_time':None,'publisher_last_edit_display_time':None,'matched_percent':None,'status':'unknown'}
 if rec['http_status']==200 and rec['local_path'] and first:
  soup=BeautifulSoup((root/rec['local_path']).read_bytes(),'html.parser');full=soup.get_text(' ',strip=True)
  if month=='2023-07':
   initial=re.search(r'기사등록\s*(2023/07/21)\s*(\d{2}:\d{2}:\d{2})',full);edited=re.search(r'최종수정\s*(2023/07/21)\s*(\d{2}:\d{2}:\d{2})',full);article=full.split('◎공감언론')[0]
  else:
   initial=re.search(r'기사입력\s*:\s*(2023년08월21일)\s*(\d{2}:\d{2})',full);edited=re.search(r'최종수정\s*:\s*(2023년08월21일)\s*(\d{2}:\d{2})',full);article=full.split('저작권자(c)')[0]
  contexts=[article[m.start():m.start()+80] for m in re.finditer('반도체',article)];expected_abs=abs(case['expected_reported_export_YoY_pct']);matching=[s for s in contexts if re.search(r'\(\s*[-△]?\s*'+re.escape(f'{expected_abs:.1f}')+r'\s*%\s*\)',s)]
  if initial and edited and len(matching)>=1 and all(s==matching[0] for s in matching):
   w['publisher_initial_display_time']=' '.join(initial.groups());w['publisher_last_edit_display_time']=' '.join(edited.groups());w['matched_percent']=case['expected_reported_export_YoY_pct'];w['status']='dated_current_publisher_page_supports_then_public_numeric_value'
  else:w['parse_diagnostic']={'initial':bool(initial),'edited':bool(edited),'contexts_with_expected_value':len(matching),'unique_matching_contexts':len(set(matching))}
 cases.append(w)
result={'status':'two_2023_public_numeric_witnesses_source_only' if all(x['status'].startswith('dated_') for x in cases) else 'some_public_numeric_witness_unknown','plan_sha256':sha(card/'plan.json'),'cases':cases,'GETs_started':state['GETs_started'],'qualified_current_value_witnesses':sum(x['status'].startswith('dated_') for x in cases),'official_first_public_bytes_qualified':False,'all_105_months_qualified':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
settings=_settings(root);files=sorted(p for p in snap.iterdir() if p.is_file());entries={str(p.relative_to(root)):sha(p) for p in files}
with tempfile.TemporaryDirectory(prefix='kcs-public-witness-') as tmp:
 archive=pathlib.Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='kcs-public-witness-restore-') as back:
  fetched=pathlib.Path(back)/'source.tar.gz'
  with fetched.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True,timeout=90)
  assert sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as t:
   members={m.name:m for m in t.getmembers() if m.isfile()};assert set(members)==set(entries)
   for rel,h in entries.items():assert hashlib.sha256(t.extractfile(members[rel]).read()).hexdigest()==h
source_evidence={**receipt,'nas_path':remote,'files':entries,'cold_restore_verified':True};(card/'source_evidence.json').write_text(json.dumps(source_evidence,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_evidence=source_evidence);rec.save_objects(artifact_path='analysis',cases=cases,result=result,calendar_dates=calendar_dates);rec.log_metrics(witnesses=result['qualified_current_value_witnesses'],GETs=state['GETs_started'],model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');record_evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(record_evidence['nas_path'])).split()[0]==record_evidence['sha256']
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_evidence':'source_evidence.json','source_NAS_sha256':digest,'source_NAS_cold_restore_verified':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':record_evidence,'NAS_readback':{'sha256':record_evidence['sha256'],'remote_hash_verified':True},'qualified_current_value_witnesses':result['qualified_current_value_witnesses'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']},'limitations':['Self-declared publisher timestamps and current HTML do not certify immutable 2023 page bytes.','Two monthly numeric witnesses do not qualify 105 historical first-public official attachments.','No Stage1 information screen, 0bp account or SOXX strategy success.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':result['status'],'witnesses':result['qualified_current_value_witnesses'],'source_NAS_sha256':digest,'recorder_id':rid},ensure_ascii=False))


2026/10/23 13:49:31 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[757:MainThread](2026-10-23 13:49:43,973) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[757:MainThread](2026-10-23 13:49:43,979) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[757:MainThread](2026-10-23 13:49:43,980) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[757:MainThread](2026-10-23 13:49:46,276) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name kcs_early20_2023_jul_aug_public_value_witness_20261023.


[757:MainThread](2026-10-23 13:49:46,279) INFO - qlib.workflow - [exp.py:258] - Experiment 256758844114966158 starts running ...


[757:MainThread](2026-10-23 13:49:46,445) INFO - qlib.workflow - [recorder.py:345] - Recorder 9d17c7aa44824b2fb91368866910ee69 starts running under Experiment 256758844114966158 ...


[757:MainThread](2026-10-23 13:49:47,118) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "two_2023_public_numeric_witnesses_source_only", "witnesses": 2, "source_NAS_sha256": "58a95b7c620da441f4c6a9f8a1c8189f6fc211e05ca8f1194b036ee8aa1f7058", "recorder_id": "9d17c7aa44824b2fb91368866910ee69"}


# 结论：2023年7/8月 KCS 半导体出口数值的公开时间见证

父方案保持冻结完整 Stage1 f696+b472、0bp 额外滑点和 IBKR Tiered 费用；本卡只做来源核验，0 SOXX 收益读取、0 拟合、0 账户。

官方当期 HWP 原件的半导体出口同比，2023-07 为 **−35.4%**、2023-08 为 **−24.7%**。两篇独立当期报道的当前页面逐值相同：Newsis 自标 2023-07-21 09:12:43 刊出、09:22:05 修改；Newspim 自标 2023-08-21 09:35 刊出、10:09 修改。按冻结的“文章日期之后首个美国 Qlib 交易日”时钟，两值最早分别进入 2023-07-24 与 2023-08-22 的候选信息集。两份当前网页原字节、HTTP 收据和原始来源引用已 NAS 冷恢复；Qlib Recorder `9d17c7aa44824b2fb91368866910ee69` 的选定读回也一致。

**决定：仅两个月的公开数值时间线得到独立旁证。** 出版方页面的日期和修改时间为当前页面自述，没有 2023 年不可变网页快照；不能证明官方 HWP 的首次公开字节，也不能外推到全部 105 个月。原始同比仍受工作日差异影响。只有继续核对版本、时间与接收任务后，才能将信息放入完整 Stage1 的 0bp 费用后 SOXX 账户；本卡没有策略胜出证据。
